In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer
import torch
import pandas as pd
from datasets import Dataset
from transformers import DataCollatorWithPadding
from sklearn.metrics import classification_report

In [5]:
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(smaller_path)

df.rename(columns={'label': 'labels'}, inplace=True)

In [ ]:
model_id = "answerdotai/ModernBERT-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=1024
    )

hf_dataset = Dataset.from_pandas(df)

# 2. Apply the tokenization
# (batched=True is crucial here so it processes chunks of text at once)
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 5230.29it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 15000/15000 [00:01<00:00, 9765.62 examples/s] 


In [ ]:
print(df['labels'].unique())

[1 0]


In [1]:
#LOADING MODEL, PRETRAINED
from transformers import AutoModelForSequenceClassification, AutoTokenizer

save_path = "./modernBERT-final"

my_model = AutoModelForSequenceClassification.from_pretrained(save_path)
my_tokenizer = AutoTokenizer.from_pretrained(save_path)

c:\VSCode Python\FakeNewsDetection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 9117.33it/s]


In [6]:
from datasets import load_dataset

# Replace with your actual file path and format
# Assuming your text column is named "text" and labels are "label"
dataset = load_dataset("csv", data_files={"test": big_path})

def tokenize_function(examples):
    # Adjust "text" to match the column name in your dataset
    return my_tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply the tokenizer to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

import evaluate
import numpy as np

# Load the accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

from transformers import Trainer

# Initialize the Trainer strictly for evaluation
trainer = Trainer(
    model=my_model,
    eval_dataset=tokenized_dataset["test"], 
    compute_metrics=compute_metrics,
)

# Run the test
results = trainer.evaluate()
print(results)

Generating test split: 288260 examples [00:01, 152234.73 examples/s]
Map: 100%|██████████| 288260/288260 [09:05<00:00, 528.72 examples/s]


KeyboardInterrupt: 

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./modernbert-fake-news",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.085408,0.075156,0.980000
2,0.043901,0.086478,0.981000
3,0.008047,0.136907,0.980000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]


TrainOutput(global_step=4500, training_loss=0.05833432330025567, metrics={'train_runtime': 1953.4665, 'train_samples_per_second': 18.429, 'train_steps_per_second': 2.304, 'total_flos': 1.0921624398317472e+16, 'train_loss': 0.05833432330025567, 'epoch': 3.0})

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
test_results = trainer.predict(split_datasets["test"])

predicted_labels = np.argmax(test_results.predictions, axis=-1)
actual_labels = test_results.label_ids

print(classification_report(actual_labels, predicted_labels, target_names=["real", "fake"]))

              precision    recall  f1-score   support

        real       0.99      0.97      0.98      1489
        fake       0.97      0.99      0.98      1511

    accuracy                           0.98      3000
   macro avg       0.98      0.98      0.98      3000
weighted avg       0.98      0.98      0.98      3000



In [ ]:
save_path = "./modernBERT-final"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


('./modernBERT-final\\tokenizer_config.json',
 './modernBERT-final\\tokenizer.json')

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model.to(device)

def predict_article(text):
    inputs = my_tokenizer(
        text, 
        return_tensors="pt",
        truncation=True, 
        max_length=512
    )
    
    inputs = {key: value.to(device) for key, value in inputs.items()}

    my_model.eval() 
    with torch.no_grad():
        outputs = my_model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, axis=-1).item()
    
    label_map = {0: "real", 1: "fake"}
    final_prediction = label_map[predicted_id]
    
    return final_prediction

real_text = "Popular streamer Clavicular framemogged by ASU frat leader."
prediction = predict_article(real_text)

print(f"The model predicts the real text as: {prediction}")

fake_text = "Scientists discover that the moon is made out of cheese."
prediction = predict_article(fake_text)

print(f"The model predicts the fake text as: {prediction}")

The model predicts the real text as: real
The model predicts the fake text as: fake
